In [1]:
import json
import pandas as pd
from openpyxl.styles import PatternFill, Alignment, Font

# --------------------------------------------
# 1. Path to insights.json 
# --------------------------------------------
json_path = "/Users/ousmane/Desktop/2025 - Documentations/Python/Economics/genesis-research/app/data/insights.json"

# --------------------------------------------
# 2. Read JSON into Python
# --------------------------------------------
with open(json_path, "r") as f:
    data = json.load(f)

# --------------------------------------------
# 3. Build rows (clean SUMMARY) + keep categoryColor
# --------------------------------------------
rows = []
category_colors = []

for item in data:
    raw_summary = item.get("summary", "") or ""
    clean_summary = raw_summary.replace("**", "").strip() if isinstance(raw_summary, str) else ""

    rows.append({
        "ID": item.get("id", ""),
        "DATE": item.get("date", ""),
        "CATEGORY": item.get("category", ""),
        "TITLE": item.get("title", ""),
        "SUMMARY": clean_summary,
    })

    category_colors.append(item.get("categoryColor", ""))

df = pd.DataFrame(rows)

# --------------------------------------------
# 4. Save to Excel with formatting
# --------------------------------------------
output_path = "/Users/ousmane/Desktop/2025 - Documentations/Python/Economics/genesis-research/app/data/insights.xlsx"

COLOR_MAP = {
    "bg-emerald-600": "059669",
    "bg-purple-600": "7C3AED",
    "bg-yellow-600": "CA8A04",
    "bg-gray-500": "6B7280",
}

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="Insights")

    sheet = writer.sheets["Insights"]

    # ---- 4a. Autofit columns (initial) ----
    for col in sheet.columns:
        max_length = 0
        column_letter = col[0].column_letter

        for cell in col:
            try:
                cell_value = str(cell.value) if cell.value is not None else ""
                if len(cell_value) > max_length:
                    max_length = len(cell_value)
            except Exception:
                pass

        sheet.column_dimensions[column_letter].width = max_length + 2

    # ---- 4b. SUMMARY column: wrap & justify ----
    summary_col = "E"
    sheet.column_dimensions[summary_col].width = 80

    for row in range(2, sheet.max_row + 1):
        cell = sheet[f"{summary_col}{row}"]
        cell.alignment = Alignment(
            wrap_text=True,
            horizontal="justify",
            vertical="top"
        )

    # ---- 4c. CATEGORY column: colored bg + light gray text ----
    category_col = "C"

    for row_idx, cat_color in enumerate(category_colors, start=2):
        cell = sheet[f"{category_col}{row_idx}"]
        fill_color = COLOR_MAP.get(cat_color, "4B5563")  # default gray
        cell.fill = PatternFill(
            start_color=fill_color,
            end_color=fill_color,
            fill_type="solid",
        )
        # Light gray text instead of pure white
        cell.font = Font(color="E5E7EB", bold=True, size=10)
        cell.alignment = Alignment(horizontal="center", vertical="center")

    # ---- 4d. Global font size 10 ----
    for row in sheet.iter_rows(
        min_row=1, max_row=sheet.max_row, min_col=1, max_col=5
    ):
        for cell in row:
            f = cell.font or Font()
            cell.font = Font(
                name=f.name,
                size=10,
                bold=f.bold,
                color=f.color
            )

    # ---- 4e. Make ID, DATE, TITLE columns bold (all rows) ----
    for col_letter in ["A", "B", "D"]:  # ID, DATE, TITLE
        for row in range(2, sheet.max_row + 1):
            cell = sheet[f"{col_letter}{row}"]
            f = cell.font or Font()
            cell.font = Font(
                name=f.name,
                size=10,
                bold=True,
                color=f.color
            )

    # ---- 4f. Header row bold (kept) ----
    for cell in sheet[1]:
        f = cell.font or Font()
        cell.font = Font(
            name=f.name,
            size=10,
            bold=True,
            color=f.color
        )

print("Excel file created successfully at:")
print(output_path)


Excel file created successfully at:
/Users/ousmane/Desktop/2025 - Documentations/Python/Economics/genesis-research/app/data/insights.xlsx
